In [10]:
# Environment Check
import sys
from pathlib import Path

python_version = sys.version_info
print(f"Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
print(f"Environment: {sys.executable}")

if python_version >= (3, 12):
    print("✓ Python version compatible")
else:
    print("✗ Need Python 3.12+")
    # exit()

Python Version: 3.12.3
Environment: d:\RAG\agentic-rag-research-assistant\.venv\Scripts\python.exe
✓ Python version compatible


In [11]:
from pathlib import Path

# Find Project Root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"✓ Project root: {project_root}")
else:
    raise FileNotFoundError("X Missing compose.yml - check directory structure")

✓ Project root: d:\RAG\agentic-rag-research-assistant


In [12]:
# Check Docker
import subprocess

try:
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=5, shell=True)
    if result.returncode == 0:
        print(f"✓ Docker: {result.stdout}")
    else:
        print("✗ Docker: Not working")
        # exit()
except:
    print("✗ Docker: Not found")
    # exit()

✓ Docker: Docker version 29.6.2, build dfc4efb



In [13]:
# Check Docker Compose
try:
    result = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"✓ Docker Compose: {result.stdout.split()[3]}")
    else:
        print("✗ Docker Compose: Not working")
        # exit()
except:
    print("✗ Docker Compose: Not found")
    # exit()

✓ Docker Compose: v5.3.1


In [14]:
# Check UV Package Manager
try:
    result = subprocess.run(["uv", "--version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"✓ UV: {result.stdout.strip()}")
        print("\n✓ All required software ready!")
    else:
        print("✗ UV: Not working")
        exit()
except:
    print("✗ UV: Not found")

✓ UV: uv 0.12.2 (46ead6098 2026-08-05 x86_64-pc-windows-msvc)

✓ All required software ready!


# Start Services

### Start services: docker compose up -d (Runs them in detached/background mode)
### Stop and pause services: docker compose stop

In [58]:
# Check Docker Running
try:
    result = subprocess.run(["docker", "info"], capture_output=True, timeout=5)
    if result.returncode == 0:
        print("✓ Docker is running")
    else:
        print("✗ Docker not running - start Docker Desktop")
        # exit()
except:
    print("✗ Docker daemon not accessible")
    # exit()

✓ Docker is running


In [59]:
# Check Current Containers
import json
import subprocess # Ensure subprocess is imported

try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        encoding='utf-8', # Fixed: Forces UTF-8 decoding instead of text=True
        errors='replace',  # Visual Anchor: Prevents crashes if there are still corrupt characters
        timeout=10
    )
    
    if result.returncode == 0 and result.stdout.strip():
        print("Current containers:")
        for line in result.stdout.strip().split('\n'):
            if line.strip():
                try:
                    container = json.loads(line)
                    service = container.get('Service', 'unknown')
                    state = container.get('State', 'unknown')
                    print(f"  • {service}: {state}")
                except:
                    pass
    else:
        print("No containers running")
        
except Exception as e:
    print("Could not check containers")

Current containers:
  • airflow: running
  • api: running
  • clickhouse: running
  • opensearch-dashboards: running
  • langfuse: running
  • langfuse-postgres: running
  • ollama: running
  • opensearch: running
  • postgres: running
  • redis: running


In [63]:
# Service Health Check
EXPECTED_SERVICES = {
    'api': 'FastAPI REST API server',
    'postgres': 'PostgreSQL database',
    'opensearch': 'OpenSearch search engine', 
    'opensearch-dashboards': 'OpenSearch web dashboard',
    'ollama': 'Local LLM inference server',
    'airflow': 'Workflow automation (optional - may be off)'
}

try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        encoding='utf-8', # Fixed: Forces UTF-8 decoding instead of text=True
        errors='replace',  # Visual Anchor: Prevents crashes if there are still corrupt characters
        timeout=15
    )
    
    if result.returncode == 0:
        print("SERVICE STATUS")
        print("=" * 70)
        print(f"{'Service':<20} {'State':<15} {'Status':<15} {'Notes'}")
        print("-" * 70)
    else:
        print("Could not get service status")
        exit()
        
except Exception as e:
    print(f"Error checking services: {e}")
    exit()

# Parse Service Status
found_services = set()
service_states = {}

if result.stdout.strip():
    for line in result.stdout.strip().split('\n'):
        if line.strip():
            try:
                container = json.loads(line)
                service = container.get('Service', 'unknown')
                state = container.get('State', 'unknown')
                health = container.get('Health', 'no check')
                
                found_services.add(service)
                service_states[service] = {'state': state, 'health': health}
                
                if state == 'running' and health in ['healthy', 'no check']:
                    indicator = "✓"
                    notes = "Ready"
                elif state == 'running' and health == 'unhealthy':
                    indicator = "⚠"
                    notes = "Starting up..."
                elif state == 'exited':
                    indicator = "✗"
                    notes = "Failed to start"
                else:
                    indicator = "?"
                    notes = f"Status: {state}"
                
                print(f"{indicator} {service:<18} {state:<14} {health:<14} {notes}")
                
            except json.JSONDecodeError:
                pass

SERVICE STATUS
Service              State           Status          Notes
----------------------------------------------------------------------
? airflow            running        starting       Status: running
✓ api                running        healthy        Ready
✓ clickhouse         running        healthy        Ready
✓ opensearch-dashboards running        healthy        Ready
⚠ langfuse           running        unhealthy      Starting up...
✓ langfuse-postgres  running        healthy        Ready
✓ ollama             running        healthy        Ready
✓ opensearch         running        healthy        Ready
✓ postgres           running        healthy        Ready
✓ redis              running        healthy        Ready


In [64]:
# Check Missing Services
missing_services = set(EXPECTED_SERVICES.keys()) - found_services

if missing_services:
    print("\nMISSING SERVICES:")
    print("-" * 70)
    for service in missing_services:
        description = EXPECTED_SERVICES[service]
        if service == 'airflow':
            print(f"⚠ {service:<18} not running    {'(Optional)':<14} {description}")
        else:
            print(f"✗ {service:<18} not running    {'Required':<14} {description}")

failed_services = [s for s, info in service_states.items() 
                  if info['state'] in ['exited', 'restarting'] or info['health'] == 'unhealthy']

if failed_services:
    print(f"\nTROUBLESHOOTING:")
    for service in failed_services:
        print(f"   docker compose logs {service}")
elif missing_services and 'airflow' not in missing_services:
    print(f"\nACTION NEEDED:")
    print("Start missing services: docker compose up -d")


TROUBLESHOOTING:
   docker compose logs langfuse


FastAPI - REST API Service

In [43]:
# Test FastAPI Health
import requests

try:
    response = requests.get("http://localhost:8000/api/v1/health", timeout=5)
    if response.status_code == 200:
        data = response.json()
        print("✓ FastAPI is responding")
        print(f"Status: {data.get('status', 'unknown')}")
    else:
        print(f"⚠ API returned status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("✗ API not responding - wait 1-2 minutes")
except Exception as e:
    print(f"✗ API test error: {e}")

✓ FastAPI is responding
Status: ok


In [44]:
# PRODUCTION INSIGHTS
print("\n" + "="*60)
print("  PRODUCTION INSIGHT (Online Sessions Only)")
print("="*60)
print("❓ How are they scaled?")
print("❓ What are the bottlenecks?")
print("❓ How are they monitored and managed?")
print("❓ How are they integrated with other systems?")
print("❓ What are the best practices for using these systems?")
print("❓ How are these systems used and deployed in production?")
print("❓ How are they tested? in terms of load and performance?")
print("→ Learn these production secrets in our online walkthrough sessions!")
print("="*60)


  PRODUCTION INSIGHT (Online Sessions Only)
❓ How are they scaled?
❓ What are the bottlenecks?
❓ How are they monitored and managed?
❓ How are they integrated with other systems?
❓ What are the best practices for using these systems?
❓ How are these systems used and deployed in production?
❓ How are they tested? in terms of load and performance?
→ Learn these production secrets in our online walkthrough sessions!


###### echo '{"admin": "<password>"}' > simple_auth_manager_passwords.json.generated 
###### docker cp simple_auth_manager_passwords.json.generated $(docker ps -qf "name=rag-airflow"):/opt/airflow/simple_auth_manager_passwords.json.generated
###### docker exec -u 0 -it $(docker ps -qf "name=rag-airflow") chown airflow:airflow /opt/airflow/simple_auth_manager_passwords.json.generated
###### docker restart $(docker ps -qf "name=rag-airflow") 



In [70]:
# Get Airflow Password
import json
from pathlib import Path

password_file = project_root / "airflow" / "simple_auth_manager_passwords.json.generated"

try:
    if password_file.exists():
        with open(password_file, 'r') as f:
            data = json.load(f)
            password = data.get("admin")
        print(f"✓ Airflow password: {password}")
    else:
        print(f"⚠ Password file not found")
        password = None
except Exception as e:
    print(f"✗ Could not read password: {e}")
    password = None

✗ Could not read password: Expecting value: line 1 column 1 (char 0)


In [66]:
# Test Airflow Health
try:
    response = requests.get("http://localhost:8080/api/v1/health", timeout=5)
    if response.status_code == 200:
        print("✓ Airflow is healthy")
        
        if password:
            print(f"\nAirflow Login:")
            print(f"URL: http://localhost:8080")
            print(f"Username: admin")
            print(f"Password: {password}")
    else:
        print(f"⚠ Airflow returned: {response.status_code}")
        
except requests.exceptions.ConnectionError:
    print("✗ Airflow not responding - wait 2-3 minutes")
except Exception as e:
    print(f"✗ Airflow test error: {e}")

✓ Airflow is healthy
